# ⚡ Module 3.3: The Bellman Equations

## The Recursive Heart of Reinforcement Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_03_RL_Mathematical_Foundations/03_Bellman_Equations/bellman_equations.ipynb)

---

### 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Derive the Bellman Expectation Equation** from first principles
2. **Understand the Bellman Optimality Equation** and how it differs from the expectation form
3. **Work with both V and Q function versions** of the Bellman equations
4. **Solve numerical examples step by step** using matrix methods and iteration
5. **Visualize the solution of small MDPs** including grid worlds with optimal policies

---

### 📋 Prerequisites

- Module 3.1: Probability for RL (expectation, conditional probability)
- Module 3.2: Markov Decision Processes (states, actions, transitions, rewards)

---

In [ ]:
!pip install numpy matplotlib networkx -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
np.random.seed(42)
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset for RL demonstrations
import torchvision
import numpy as np

# MNIST as discrete image states for MDP demonstrations
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True)
# Use small subset as "states" in our MDP
sample_states = [np.array(mnist[i][0]) for i in range(100)]
print(f"✅ MNIST loaded: Using {len(sample_states)} real images as MDP states")
print("   Each state is a 28x28 grayscale image = 784-dimensional state vector")

## Deep Derivation: Bellman Equation from First Principles

### Step 1: Definition of Value Function
$$V^\pi(s) = E_\pi\left[\sum_{t=0}^{\infty} \gamma^t R_{t+1} \mid S_0 = s\right]$$

### Step 2: Expand the Sum
$$V^\pi(s) = E_\pi\left[R_1 + \gamma R_2 + \gamma^2 R_3 + \cdots \mid S_0 = s\right]$$

### Step 3: Factor Out First Reward
$$V^\pi(s) = E_\pi\left[R_1 + \gamma \left(R_2 + \gamma R_3 + \cdots\right) \mid S_0 = s\right]$$

### Step 4: Recognize Inner Sum as $V^\pi(S_1)$
$$V^\pi(s) = E_\pi\left[R_1 + \gamma V^\pi(S_1) \mid S_0 = s\right]$$

### Step 5: Expand Expectation Over Actions and Next States
$$V^\pi(s) = \sum_{a \in \mathcal{A}} \pi(a|s) \sum_{s' \in \mathcal{S}} P(s'|s,a) \left[R(s,a,s') + \gamma V^\pi(s')\right]$$

**This is the Bellman Expectation Equation.** $\blacksquare$

### Step 6: Bellman Optimality (replace $\pi$ with $\max$)
The optimal value satisfies:
$$V^*(s) = \max_{a \in \mathcal{A}} \sum_{s'} P(s'|s,a) \left[R(s,a,s') + \gamma V^*(s')\right]$$

### Step 7: Contraction Mapping Proof (Why Value Iteration Converges)

**Define Bellman operator:** $(\mathcal{T}V)(s) = \max_a \sum_{s'} P(s'|s,a)[R + \gamma V(s')]$

**Claim:** $\mathcal{T}$ is a $\gamma$-contraction under $\|\cdot\|_\infty$

**Proof:**
$$\|\mathcal{T}V_1 - \mathcal{T}V_2\|_\infty = \max_s \left|\max_a \sum_{s'} P(s'|s,a)\gamma[V_1(s') - V_2(s')] \right|$$
$$\leq \max_s \max_a \sum_{s'} P(s'|s,a) \gamma |V_1(s') - V_2(s')|$$
$$\leq \gamma \|V_1 - V_2\|_\infty \cdot \max_s \max_a \sum_{s'} P(s'|s,a) = \gamma \|V_1 - V_2\|_\infty$$

Since $\gamma < 1$, by **Banach Fixed-Point Theorem**, $\mathcal{T}$ has a unique fixed point $V^*$ and iteration $V_{k+1} = \mathcal{T}V_k$ converges to $V^*$ at rate $O(\gamma^k)$. $\blacksquare$

### Step 8: Q-Function Version
$$Q^*(s,a) = \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma \max_{a'} Q^*(s', a')\right]$$

**Relationship:** $V^*(s) = \max_a Q^*(s,a)$ and $\pi^*(s) = \arg\max_a Q^*(s,a)$

---

## 🔄 Section 1: The Key Insight — Recursion

### Why Recursion Matters

Recall the definition of **return** from Module 3.2:

$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

Here's the crucial observation — we can **factor out** $\gamma$:

$$G_t = R_{t+1} + \gamma \left( R_{t+2} + \gamma R_{t+3} + \cdots \right) = R_{t+1} + \gamma G_{t+1}$$

This is **recursion**: the return at time $t$ depends on the return at time $t+1$.

### 💡 The Bootstrapping Idea

Since $V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]$, the recursive nature of $G_t$ means:

> **"I can estimate the value of the current state using my estimate of the value of the next state."**

This is called **bootstrapping** — we "pull ourselves up by our own bootstraps" by using our own estimates to improve our estimates. This is the foundational idea behind all Bellman equations and most RL algorithms.

| Approach | Idea | Analogy |
|----------|------|---------|
| **No bootstrapping** | Wait until the end to compute return | Grade an exam after answering ALL questions |
| **Bootstrapping** | Use current estimates of future values | Estimate your exam score after each question |

In [ ]:
# === Demonstrating the Recursive Nature of Returns ===

rewards = [2, 3, 5, 1, 4, 2, 3]
gamma = 0.9

print("="*60)
print("🔄 RECURSIVE NATURE OF RETURNS")
print("="*60)
print(f"\nReward sequence: {rewards}")
print(f"Discount factor γ = {gamma}")

# Method 1: Direct computation from definition
# G_t = sum_{k=0}^{T-t-1} gamma^k * R_{t+k+1}
T = len(rewards)
returns_direct = np.zeros(T)
for t in range(T):
    G = 0
    for k in range(t, T):
        G += (gamma ** (k - t)) * rewards[k]
    returns_direct[t] = G

# Method 2: Recursive computation  G_t = R_{t+1} + gamma * G_{t+1}
returns_recursive = np.zeros(T)
returns_recursive[T - 1] = rewards[T - 1]
for t in range(T - 2, -1, -1):
    returns_recursive[t] = rewards[t] + gamma * returns_recursive[t + 1]

print("\n" + "-"*60)
print(f"{'Time t':<10} {'Direct G_t':<15} {'Recursive G_t':<15} {'Match?':<10}")
print("-"*60)
for t in range(T):
    match = "✅" if np.isclose(returns_direct[t], returns_recursive[t]) else "❌"
    print(f"{t:<10} {returns_direct[t]:<15.4f} {returns_recursive[t]:<15.4f} {match:<10}")

print("\n💡 Key insight: Both methods give IDENTICAL results!")
print("   The recursive form G_t = R_{t+1} + γ·G_{t+1} is much more efficient.")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(range(T), rewards, color='steelblue', alpha=0.8, edgecolor='navy')
axes[0].set_xlabel('Time Step t', fontsize=12)
axes[0].set_ylabel('Reward R_t', fontsize=12)
axes[0].set_title('📊 Reward Sequence', fontsize=14, fontweight='bold')
axes[0].set_xticks(range(T))

axes[1].plot(range(T), returns_direct, 'o-', color='crimson', linewidth=2,
             markersize=8, label='Direct computation')
axes[1].plot(range(T), returns_recursive, 's--', color='forestgreen', linewidth=2,
             markersize=8, label='Recursive computation', alpha=0.7)
axes[1].set_xlabel('Time Step t', fontsize=12)
axes[1].set_ylabel('Return G_t', fontsize=12)
axes[1].set_title('📈 Returns (Both Methods Agree)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].set_xticks(range(T))

plt.tight_layout()
plt.show()
print("\n🔑 The return DECREASES at later time steps because fewer future rewards remain.")

---

## 📐 Section 2: Bellman Expectation Equation for $V^\pi$

### Full Step-by-Step Derivation

We start from the definition of the state-value function and derive the Bellman equation through careful application of probability rules.

---

**Step 1:** Start from the definition of $V^\pi(s)$:

$$V^\pi(s) = \mathbb{E}_\pi\left[G_t \mid S_t = s\right]$$

---

**Step 2:** Substitute the recursive form of the return $G_t = R_{t+1} + \gamma G_{t+1}$:

$$V^\pi(s) = \mathbb{E}_\pi\left[R_{t+1} + \gamma G_{t+1} \mid S_t = s\right]$$

---

**Step 3:** Apply **linearity of expectation** to split the sum:

$$V^\pi(s) = \mathbb{E}_\pi\left[R_{t+1} \mid S_t = s\right] + \gamma \, \mathbb{E}_\pi\left[G_{t+1} \mid S_t = s\right]$$

---

**Step 4:** Apply the **law of total expectation** — condition on action $a$ and next state $s'$:

$$\mathbb{E}_\pi\left[R_{t+1} \mid S_t = s\right] = \sum_{a} \pi(a|s) \, R(s,a)$$

where $R(s,a) = \mathbb{E}[R_{t+1} \mid S_t = s, A_t = a]$ is the expected immediate reward.

---

**Step 5:** For the second term, condition on $a$ and $s'$:

$$\mathbb{E}_\pi\left[G_{t+1} \mid S_t = s\right] = \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a) \, \mathbb{E}_\pi\left[G_{t+1} \mid S_{t+1} = s'\right]$$

---

**Step 6:** Recognize that $\mathbb{E}_\pi[G_{t+1} \mid S_{t+1} = s'] = V^\pi(s')$ by the **Markov property**:

$$\mathbb{E}_\pi\left[G_{t+1} \mid S_t = s\right] = \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a) \, V^\pi(s')$$

---

**Step 7:** Combine Steps 4 and 6 to get the **Bellman Expectation Equation for $V^\pi$**:

$$\boxed{V^\pi(s) = \sum_{a} \pi(a|s) \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a) \, V^\pi(s') \right]}$$

---

### 🧮 Matrix Form

For a finite MDP with $n$ states, we can write the Bellman equation as a **linear system**:

$$\mathbf{V}^\pi = \mathbf{R}^\pi + \gamma \mathbf{P}^\pi \mathbf{V}^\pi$$

Solving for $\mathbf{V}^\pi$:

$$(\mathbf{I} - \gamma \mathbf{P}^\pi) \mathbf{V}^\pi = \mathbf{R}^\pi$$

$$\boxed{\mathbf{V}^\pi = (\mathbf{I} - \gamma \mathbf{P}^\pi)^{-1} \mathbf{R}^\pi}$$

where $\mathbf{P}^\pi$ is the state transition matrix under policy $\pi$, and $\mathbf{R}^\pi$ is the expected reward vector under $\pi$.

In [ ]:
# === Bellman Expectation Equation for V^π: 3-State MDP ===

print("="*60)
print("📐 BELLMAN EXPECTATION EQUATION FOR V^π")
print("="*60)

# Define a 3-state MDP
n_states = 3
n_actions = 2
gamma = 0.9
state_names = ['s0', 's1', 's2']
action_names = ['a0', 'a1']

# Transition probabilities P(s'|s,a) — shape: (n_states, n_actions, n_states)
P = np.zeros((n_states, n_actions, n_states))
# From s0:
P[0, 0] = [0.2, 0.6, 0.2]  # action a0
P[0, 1] = [0.1, 0.3, 0.6]  # action a1
# From s1:
P[1, 0] = [0.3, 0.4, 0.3]
P[1, 1] = [0.5, 0.2, 0.3]
# From s2:
P[2, 0] = [0.1, 0.2, 0.7]
P[2, 1] = [0.4, 0.4, 0.2]

# Rewards R(s,a) — shape: (n_states, n_actions)
R = np.array([
    [5.0, 10.0],   # s0
    [-1.0, 2.0],   # s1
    [0.0, 3.0]     # s2
])

# Policy π(a|s) — shape: (n_states, n_actions)
# Let's use a stochastic policy
policy = np.array([
    [0.4, 0.6],   # s0: 40% a0, 60% a1
    [0.7, 0.3],   # s1: 70% a0, 30% a1
    [0.5, 0.5]    # s2: 50% a0, 50% a1
])

print("\n🎲 MDP Definition:")
print(f"  States: {state_names}")
print(f"  Actions: {action_names}")
print(f"  γ = {gamma}")

print("\n📋 Policy π(a|s):")
for s in range(n_states):
    probs = ', '.join([f'π({action_names[a]}|{state_names[s]})={policy[s,a]:.1f}' for a in range(n_actions)])
    print(f"  {state_names[s]}: {probs}")

# --- Method 1: Matrix solution V = (I - γ P_π)^{-1} R_π ---

# Compute P_π(s'|s) = sum_a π(a|s) P(s'|s,a)
P_pi = np.zeros((n_states, n_states))
for s in range(n_states):
    for a in range(n_actions):
        P_pi[s] += policy[s, a] * P[s, a]

# Compute R_π(s) = sum_a π(a|s) R(s,a)
R_pi = np.sum(policy * R, axis=1)

print("\n📊 Policy transition matrix P^π:")
print(np.round(P_pi, 3))
print(f"\n📊 Policy reward vector R^π: {np.round(R_pi, 3)}")

# Solve: V = (I - γ P_π)^{-1} R_π
I = np.eye(n_states)
V_pi_matrix = np.linalg.solve(I - gamma * P_pi, R_pi)

print("\n" + "="*40)
print("✅ Solution via Matrix Inversion:")
print("="*40)
for s in range(n_states):
    print(f"  V^π({state_names[s]}) = {V_pi_matrix[s]:.4f}")

# --- Method 2: Iterative Policy Evaluation (verification) ---
V_iter = np.zeros(n_states)
n_iterations = 200
tolerance = 1e-8

for i in range(n_iterations):
    V_new = np.zeros(n_states)
    for s in range(n_states):
        for a in range(n_actions):
            V_new[s] += policy[s, a] * (R[s, a] + gamma * np.dot(P[s, a], V_iter))
    if np.max(np.abs(V_new - V_iter)) < tolerance:
        print(f"\n🔄 Iterative evaluation converged in {i+1} iterations")
        break
    V_iter = V_new

print("\n✅ Verification via Iterative Policy Evaluation:")
for s in range(n_states):
    match = "✅" if np.isclose(V_pi_matrix[s], V_iter[s], atol=1e-4) else "❌"
    print(f"  V^π({state_names[s]}) = {V_iter[s]:.4f}  {match}")

print("\n💡 Both methods produce the same result — the Bellman equation is consistent!")

---

## 🎯 Section 3: Bellman Expectation Equation for $Q^\pi$

### Derivation

The **action-value function** $Q^\pi(s,a)$ can also be expressed recursively. Starting from its definition:

$$Q^\pi(s,a) = \mathbb{E}_\pi\left[G_t \mid S_t = s, A_t = a\right]$$

Expanding with $G_t = R_{t+1} + \gamma G_{t+1}$:

$$Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \, \mathbb{E}_\pi\left[G_{t+1} \mid S_{t+1} = s'\right]$$

Now, $\mathbb{E}_\pi[G_{t+1} \mid S_{t+1}=s'] = V^\pi(s') = \sum_{a'} \pi(a'|s') Q^\pi(s',a')$, so:

$$\boxed{Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \sum_{a'} \pi(a'|s') \, Q^\pi(s', a')}$$

### Key Relationships

There are two important relationships connecting $V^\pi$ and $Q^\pi$:

**1. From Q to V** (average over actions using the policy):

$$V^\pi(s) = \sum_a \pi(a|s) \, Q^\pi(s,a)$$

**2. From V to Q** (one-step lookahead):

$$Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \, V^\pi(s')$$

Together, these form a beautiful circular relationship:

$$V^\pi \xrightarrow{\text{expand with } Q} Q^\pi \xrightarrow{\text{average with } \pi} V^\pi$$

In [ ]:
# === Compute Q^π for the same 3-state MDP ===

print("="*60)
print("🎯 BELLMAN EXPECTATION EQUATION FOR Q^π")
print("="*60)

# Compute Q^π(s,a) = R(s,a) + γ Σ_{s'} P(s'|s,a) V^π(s')
Q_pi = np.zeros((n_states, n_actions))
for s in range(n_states):
    for a in range(n_actions):
        Q_pi[s, a] = R[s, a] + gamma * np.dot(P[s, a], V_pi_matrix)

print("\n📊 Q^π(s, a) Values:")
print("-"*45)
print(f"{'State':<10}", end='')
for a in range(n_actions):
    print(f"{action_names[a]:<15}", end='')
print()
print("-"*45)
for s in range(n_states):
    print(f"{state_names[s]:<10}", end='')
    for a in range(n_actions):
        print(f"{Q_pi[s, a]:<15.4f}", end='')
    print()

# Verify: V^π(s) = Σ_a π(a|s) Q^π(s,a)
print("\n✅ Verification: V^π(s) = Σ_a π(a|s) Q^π(s,a)")
print("-"*55)
for s in range(n_states):
    V_from_Q = np.dot(policy[s], Q_pi[s])
    match = "✅" if np.isclose(V_from_Q, V_pi_matrix[s]) else "❌"
    print(f"  V^π({state_names[s]}): from Q = {V_from_Q:.4f}, "
          f"direct = {V_pi_matrix[s]:.4f}  {match}")

# Heatmap of Q^π values
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(Q_pi, cmap='RdYlGn', aspect='auto')

ax.set_xticks(range(n_actions))
ax.set_xticklabels(action_names, fontsize=13)
ax.set_yticks(range(n_states))
ax.set_yticklabels(state_names, fontsize=13)
ax.set_xlabel('Action', fontsize=13)
ax.set_ylabel('State', fontsize=13)
ax.set_title('🎯 Q^π(s, a) Heatmap', fontsize=15, fontweight='bold')

for i in range(n_states):
    for j in range(n_actions):
        ax.text(j, i, f'{Q_pi[i, j]:.2f}', ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if Q_pi[i, j] < np.mean(Q_pi) else 'black')

plt.colorbar(im, ax=ax, label='Q-Value')
plt.tight_layout()
plt.show()

print("\n💡 The heatmap shows which (state, action) pairs have higher expected returns.")
print("   Greener cells = higher Q-values = better state-action pairs under policy π.")

---

## ⭐ Section 4: Bellman Optimality Equation for $V^*$

### From Expectation to Optimality

The Bellman **Expectation** Equation tells us the value of a state under a **given** policy $\pi$. But what if we want the **best possible** value?

### Definition of Optimal Value

$$V^*(s) = \max_\pi V^\pi(s) \quad \text{for all } s$$

### Derivation

Under the optimal policy, the agent always picks the **best** action. Instead of averaging over actions weighted by $\pi(a|s)$, we take the **maximum**:

$$\boxed{V^*(s) = \max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a) \, V^*(s') \right]}$$

### Key Difference: `max` vs. Weighted Sum

| Bellman Expectation | Bellman Optimality |
|--------------------|-----------------|
| $V^\pi(s) = \sum_a \pi(a{\mid}s)[\cdots]$ | $V^*(s) = \max_a [\cdots]$ |
| Weighted average over $\pi$ | Maximum over actions |
| **Linear** system of equations | **Nonlinear** system (due to max) |
| Can be solved by matrix inversion | Requires iterative methods |
| Evaluates a given policy | Finds the optimal value |

### Solving with Value Iteration

Since we can't use matrix inversion (nonlinear!), we use **value iteration**:

$$V_{k+1}(s) = \max_a \left[ R(s,a) + \gamma \sum_{s'} P(s'|s,a) \, V_k(s') \right]$$

Starting from $V_0(s) = 0$, this converges to $V^*$ as $k \to \infty$.

In [ ]:
# === Bellman Optimality Equation for V*: Value Iteration ===

print("="*60)
print("⭐ BELLMAN OPTIMALITY EQUATION FOR V*")
print("="*60)

V = np.zeros(n_states)
n_iterations = 100
tolerance = 1e-8
history = [V.copy()]

for i in range(n_iterations):
    V_new = np.zeros(n_states)
    for s in range(n_states):
        q_values = np.zeros(n_actions)
        for a in range(n_actions):
            q_values[a] = R[s, a] + gamma * np.dot(P[s, a], V)
        V_new[s] = np.max(q_values)
    
    history.append(V_new.copy())
    delta = np.max(np.abs(V_new - V))
    
    if delta < tolerance:
        print(f"\n✅ Value iteration converged in {i+1} iterations (δ < {tolerance})")
        break
    V = V_new

V_star = V_new

print("\n📊 Optimal State Values V*(s):")
print("-"*35)
for s in range(n_states):
    print(f"  V*({state_names[s]}) = {V_star[s]:.4f}")

# Compare with V^π
print("\n📊 Comparison with V^π:")
print("-"*50)
print(f"{'State':<10} {'V^π':<15} {'V*':<15} {'Improvement':<15}")
print("-"*50)
for s in range(n_states):
    improvement = V_star[s] - V_pi_matrix[s]
    print(f"{state_names[s]:<10} {V_pi_matrix[s]:<15.4f} {V_star[s]:<15.4f} {improvement:<+15.4f}")

# Plot convergence
history_arr = np.array(history)
fig, ax = plt.subplots(figsize=(12, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71']
for s in range(n_states):
    ax.plot(range(len(history_arr)), history_arr[:, s], 'o-',
            color=colors[s], linewidth=2, markersize=5,
            label=f'V*({state_names[s]})')
    ax.axhline(y=V_star[s], color=colors[s], linestyle='--', alpha=0.4)

ax.set_xlabel('Iteration', fontsize=13)
ax.set_ylabel('Value', fontsize=13)
ax.set_title('⭐ Value Iteration Convergence to V*', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 All states converge to their optimal values within a few iterations.")
print("   V* ≥ V^π always — the optimal policy can never be worse!")

---

## 🏆 Section 5: Bellman Optimality Equation for $Q^*$

### Derivation

The optimal action-value function satisfies:

$$\boxed{Q^*(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) \max_{a'} Q^*(s', a')}$$

Note the key difference from $Q^\pi$: instead of averaging over the next action using $\pi(a'|s')$, we take the **max** over $a'$.

### Relationship to $V^*$

$$V^*(s) = \max_a Q^*(s,a)$$

### Extracting the Optimal Policy

Once we have $Q^*$, the optimal policy is simply the **greedy** policy with respect to $Q^*$:

$$\pi^*(a|s) = \begin{cases} 1 & \text{if } a = \arg\max_{a'} Q^*(s,a') \\ 0 & \text{otherwise} \end{cases}$$

This is why $Q^*$ is so powerful — given $Q^*$, the agent doesn't need to know the transition dynamics to act optimally. It just picks the action with the highest $Q^*$ value!

In [ ]:
# === Compute Q* and extract optimal policy ===

print("="*60)
print("🏆 BELLMAN OPTIMALITY EQUATION FOR Q*")
print("="*60)

# Compute Q*(s,a) = R(s,a) + γ Σ_{s'} P(s'|s,a) V*(s')
Q_star = np.zeros((n_states, n_actions))
for s in range(n_states):
    for a in range(n_actions):
        Q_star[s, a] = R[s, a] + gamma * np.dot(P[s, a], V_star)

print("\n📊 Q*(s, a) Values:")
print("-"*45)
print(f"{'State':<10}", end='')
for a in range(n_actions):
    print(f"{action_names[a]:<15}", end='')
print(f"{'Best Action':<15}")
print("-"*45)
for s in range(n_states):
    print(f"{state_names[s]:<10}", end='')
    for a in range(n_actions):
        print(f"{Q_star[s, a]:<15.4f}", end='')
    best_a = np.argmax(Q_star[s])
    print(f"{action_names[best_a]:<15} ⬅️")

# Verify: V*(s) = max_a Q*(s,a)
print("\n✅ Verification: V*(s) = max_a Q*(s,a)")
print("-"*50)
for s in range(n_states):
    V_from_Q = np.max(Q_star[s])
    match = "✅" if np.isclose(V_from_Q, V_star[s]) else "❌"
    print(f"  V*({state_names[s]}): from Q* = {V_from_Q:.4f}, "
          f"direct = {V_star[s]:.4f}  {match}")

# Extract and display optimal policy
print("\n🎯 Optimal Policy π*(a|s):")
print("-"*45)
optimal_policy = np.zeros((n_states, n_actions))
for s in range(n_states):
    best_a = np.argmax(Q_star[s])
    optimal_policy[s, best_a] = 1.0
    print(f"  {state_names[s]}: π*({action_names[best_a]}|{state_names[s]}) = 1.0")

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Q* heatmap
im1 = axes[0].imshow(Q_star, cmap='RdYlGn', aspect='auto')
axes[0].set_xticks(range(n_actions))
axes[0].set_xticklabels(action_names, fontsize=12)
axes[0].set_yticks(range(n_states))
axes[0].set_yticklabels(state_names, fontsize=12)
axes[0].set_xlabel('Action', fontsize=12)
axes[0].set_ylabel('State', fontsize=12)
axes[0].set_title('Q*(s,a) Values', fontsize=14, fontweight='bold')
for i in range(n_states):
    for j in range(n_actions):
        axes[0].text(j, i, f'{Q_star[i,j]:.2f}', ha='center', va='center',
                     fontsize=13, fontweight='bold',
                     color='white' if Q_star[i,j] < np.mean(Q_star) else 'black')
plt.colorbar(im1, ax=axes[0], label='Q-Value')

# Optimal policy heatmap
im2 = axes[1].imshow(optimal_policy, cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[1].set_xticks(range(n_actions))
axes[1].set_xticklabels(action_names, fontsize=12)
axes[1].set_yticks(range(n_states))
axes[1].set_yticklabels(state_names, fontsize=12)
axes[1].set_xlabel('Action', fontsize=12)
axes[1].set_ylabel('State', fontsize=12)
axes[1].set_title('Optimal Policy π*(a|s)', fontsize=14, fontweight='bold')
for i in range(n_states):
    for j in range(n_actions):
        axes[1].text(j, i, f'{optimal_policy[i,j]:.0f}', ha='center', va='center',
                     fontsize=16, fontweight='bold',
                     color='white' if optimal_policy[i,j] > 0.5 else 'gray')
plt.colorbar(im2, ax=axes[1], label='Probability')

plt.suptitle('🏆 Q* Values and Optimal Policy', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 The optimal policy is deterministic: always pick the action with highest Q*.")

---

## 🔢 Section 6: Step-by-Step Numerical Example

### A Concrete 4-State MDP

Let's work through a complete example with specific numbers.

**States:** $\{s_0, s_1, s_2, s_3\}$ where $s_3$ is a **terminal** state

**Actions:** $\{a_0, a_1\}$

**Discount factor:** $\gamma = 0.9$

**Transition probabilities** $P(s'|s,a)$ and **Rewards** $R(s,a)$:

| State | Action | $P(s_0)$ | $P(s_1)$ | $P(s_2)$ | $P(s_3)$ | $R(s,a)$ |
|-------|--------|----------|----------|----------|----------|----------|
| $s_0$ | $a_0$ | 0.0 | 0.8 | 0.1 | 0.1 | 4.0 |
| $s_0$ | $a_1$ | 0.0 | 0.2 | 0.6 | 0.2 | 8.0 |
| $s_1$ | $a_0$ | 0.3 | 0.0 | 0.5 | 0.2 | 2.0 |
| $s_1$ | $a_1$ | 0.1 | 0.0 | 0.3 | 0.6 | 6.0 |
| $s_2$ | $a_0$ | 0.1 | 0.3 | 0.0 | 0.6 | 1.0 |
| $s_2$ | $a_1$ | 0.2 | 0.2 | 0.0 | 0.6 | 3.0 |
| $s_3$ | - | - | - | - | - | 0.0 (terminal) |

**Given policy:** $\pi(a_0|s_0) = 0.5, \pi(a_0|s_1) = 0.6, \pi(a_0|s_2) = 0.4$

In [ ]:
# === Draw the 4-state MDP as a graph ===

print("="*60)
print("🔢 STEP-BY-STEP NUMERICAL EXAMPLE")
print("="*60)

# Define the 4-state MDP
n_s = 4
n_a = 2
gamma_ex = 0.9
s_names = ['s₀', 's₁', 's₂', 's₃ (term)']
a_names = ['a₀', 'a₁']

# Transition probabilities P(s'|s,a) for non-terminal states
P_ex = np.zeros((n_s, n_a, n_s))
P_ex[0, 0] = [0.0, 0.8, 0.1, 0.1]
P_ex[0, 1] = [0.0, 0.2, 0.6, 0.2]
P_ex[1, 0] = [0.3, 0.0, 0.5, 0.2]
P_ex[1, 1] = [0.1, 0.0, 0.3, 0.6]
P_ex[2, 0] = [0.1, 0.3, 0.0, 0.6]
P_ex[2, 1] = [0.2, 0.2, 0.0, 0.6]
# s3 is terminal — all transitions stay at s3 with zero reward
P_ex[3, 0] = [0.0, 0.0, 0.0, 1.0]
P_ex[3, 1] = [0.0, 0.0, 0.0, 1.0]

R_ex = np.array([
    [4.0, 8.0],
    [2.0, 6.0],
    [1.0, 3.0],
    [0.0, 0.0]
])

# Draw MDP graph
fig, ax = plt.subplots(figsize=(12, 8))

G = nx.MultiDiGraph()
for s in range(n_s):
    G.add_node(s_names[s])

pos = {
    's₀': (0, 1),
    's₁': (2, 2),
    's₂': (2, 0),
    's₃ (term)': (4, 1)
}

# Draw nodes
node_colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=2500,
                       node_color=node_colors, edgecolors='black', linewidths=2)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=12, font_weight='bold')

# Add edge annotations
edge_info = []
for s in range(n_s - 1):
    for a in range(n_a):
        for s_next in range(n_s):
            p = P_ex[s, a, s_next]
            if p > 0:
                edge_info.append((s_names[s], s_names[s_next],
                                  f"{a_names[a]}: p={p:.1f}, r={R_ex[s,a]:.0f}"))

ax.set_title('🔢 4-State MDP Graph\n(edges show action: probability, reward)',
             fontsize=15, fontweight='bold')

# Draw key transitions with curved edges
for s in range(n_s - 1):
    for a in range(n_a):
        for s_next in range(n_s):
            p = P_ex[s, a, s_next]
            if p > 0:
                start = np.array(pos[s_names[s]])
                end = np.array(pos[s_names[s_next]])
                color = '#2980b9' if a == 0 else '#c0392b'
                alpha = min(0.3 + p * 0.7, 1.0)
                ax.annotate('', xy=end, xytext=start,
                           arrowprops=dict(arrowstyle='->', color=color,
                                          lw=1.5 * p + 0.5, alpha=alpha,
                                          connectionstyle=f'arc3,rad={0.15 * (a*2-1)}'))

legend_elements = [
    mpatches.Patch(color='#2980b9', label=f'{a_names[0]} transitions'),
    mpatches.Patch(color='#c0392b', label=f'{a_names[1]} transitions'),
]
ax.legend(handles=legend_elements, loc='lower left', fontsize=11)
ax.set_xlim(-0.8, 5.0)
ax.set_ylim(-0.8, 2.8)
ax.axis('off')

plt.tight_layout()
plt.show()

print("\n📊 Blue edges = action a₀, Red edges = action a₁")
print("   Thicker edges = higher transition probability")

In [ ]:
# === Solve V^π step by step with iterative policy evaluation ===

print("="*60)
print("📋 ITERATIVE POLICY EVALUATION — Step by Step")
print("="*60)

# Given policy
pi_ex = np.array([
    [0.5, 0.5],   # s0
    [0.6, 0.4],   # s1
    [0.4, 0.6],   # s2
    [0.5, 0.5]    # s3 (doesn't matter — terminal)
])

print("\n📋 Policy π:")
for s in range(n_s - 1):
    print(f"  {s_names[s]}: π(a₀)={pi_ex[s,0]:.1f}, π(a₁)={pi_ex[s,1]:.1f}")
print(f"  {s_names[3]}: terminal (value = 0)")
print(f"  γ = {gamma_ex}")

# Iterative policy evaluation
V_eval = np.zeros(n_s)
max_iter = 50
tol = 1e-6

print(f"\n{'Iter':<6}", end='')
for s in range(n_s):
    print(f"{'V('+s_names[s]+')':<15}", end='')
print(f"{'Max Δ':<12}")
print("-"*65)

print(f"{'0':<6}", end='')
for s in range(n_s):
    print(f"{V_eval[s]:<15.4f}", end='')
print(f"{'—':<12}")

eval_history = [V_eval.copy()]

for iteration in range(1, max_iter + 1):
    V_new = np.zeros(n_s)
    for s in range(n_s):
        if s == n_s - 1:  # terminal state
            V_new[s] = 0
            continue
        for a in range(n_a):
            V_new[s] += pi_ex[s, a] * (R_ex[s, a] + gamma_ex * np.dot(P_ex[s, a], V_eval))
    
    delta = np.max(np.abs(V_new - V_eval))
    V_eval = V_new.copy()
    eval_history.append(V_eval.copy())
    
    if iteration <= 10 or iteration % 5 == 0 or delta < tol:
        print(f"{iteration:<6}", end='')
        for s in range(n_s):
            print(f"{V_eval[s]:<15.4f}", end='')
        print(f"{delta:<12.6f}")
    
    if delta < tol:
        print(f"\n✅ Converged at iteration {iteration}!")
        break

print("\n📊 Final V^π values:")
for s in range(n_s):
    print(f"  V^π({s_names[s]}) = {V_eval[s]:.4f}")

In [ ]:
# === Solve V* using value iteration — step by step ===

print("="*60)
print("⭐ VALUE ITERATION — Step by Step")
print("="*60)

V_vi = np.zeros(n_s)
vi_history = [V_vi.copy()]

print(f"\n{'Iter':<6}", end='')
for s in range(n_s):
    print(f"{'V('+s_names[s]+')':<15}", end='')
print(f"{'Max Δ':<12} {'Best Actions':<20}")
print("-"*85)

print(f"{'0':<6}", end='')
for s in range(n_s):
    print(f"{V_vi[s]:<15.4f}", end='')
print(f"{'—':<12} {'—':<20}")

for iteration in range(1, max_iter + 1):
    V_new = np.zeros(n_s)
    best_actions = []
    
    for s in range(n_s):
        if s == n_s - 1:  # terminal
            V_new[s] = 0
            best_actions.append('-')
            continue
        q_vals = np.zeros(n_a)
        for a in range(n_a):
            q_vals[a] = R_ex[s, a] + gamma_ex * np.dot(P_ex[s, a], V_vi)
        V_new[s] = np.max(q_vals)
        best_actions.append(a_names[np.argmax(q_vals)])
    
    delta = np.max(np.abs(V_new - V_vi))
    V_vi = V_new.copy()
    vi_history.append(V_vi.copy())
    
    if iteration <= 10 or iteration % 5 == 0 or delta < tol:
        print(f"{iteration:<6}", end='')
        for s in range(n_s):
            print(f"{V_vi[s]:<15.4f}", end='')
        actions_str = ', '.join(best_actions[:3])
        print(f"{delta:<12.6f} [{actions_str}]")
    
    if delta < tol:
        print(f"\n✅ Value iteration converged at iteration {iteration}!")
        break

# Extract optimal policy
print("\n📊 Optimal Values and Policy:")
print("-"*50)
for s in range(n_s - 1):
    q_vals = np.array([R_ex[s, a] + gamma_ex * np.dot(P_ex[s, a], V_vi) for a in range(n_a)])
    best_a = np.argmax(q_vals)
    print(f"  V*({s_names[s]}) = {V_vi[s]:.4f}  →  π*({s_names[s]}) = {a_names[best_a]}")
    print(f"    Q({s_names[s]}, a₀) = {q_vals[0]:.4f}, Q({s_names[s]}, a₁) = {q_vals[1]:.4f}")

# Comparison plot
fig, ax = plt.subplots(figsize=(12, 5))
vi_arr = np.array(vi_history)
eval_arr = np.array(eval_history)

colors_4 = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
for s in range(n_s - 1):
    ax.plot(range(len(vi_arr)), vi_arr[:, s], 'o-', color=colors_4[s],
            linewidth=2, markersize=4, label=f'V*({s_names[s]})')
    ax.plot(range(len(eval_arr)), eval_arr[:, s], 's--', color=colors_4[s],
            linewidth=1.5, markersize=3, alpha=0.5, label=f'V^π({s_names[s]})')

ax.set_xlabel('Iteration', fontsize=13)
ax.set_ylabel('Value', fontsize=13)
ax.set_title('⭐ Value Iteration (solid) vs Policy Evaluation (dashed)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 V* (solid lines) converges to higher values than V^π (dashed lines) for each state.")

---

## 🗺️ Section 7: Visual MDP Solver — Grid World

### Applying Bellman Equations to a 4×4 Grid World

Grid worlds are the classic testbed for RL algorithms. Let's solve one from scratch:

- **16 states** arranged in a 4×4 grid
- **4 actions**: Up (↑), Down (↓), Left (←), Right (→)
- Moving off the grid keeps you in the same cell
- **Goal state** (bottom-right): reward = +1
- **Penalty state** (row 1, col 3): reward = −1
- **All other transitions**: reward = −0.04 (small step penalty to encourage efficiency)

In [ ]:
# === Define the 4x4 Grid World MDP ===

print("="*60)
print("🗺️ 4×4 GRID WORLD MDP")
print("="*60)

GRID_SIZE = 4
N_STATES_GW = GRID_SIZE * GRID_SIZE
N_ACTIONS_GW = 4
GAMMA_GW = 0.9

# Actions: 0=Up, 1=Down, 2=Left, 3=Right
ACTION_NAMES_GW = ['↑', '↓', '←', '→']
ACTION_DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # (row_delta, col_delta)

GOAL_STATE = (3, 3)     # bottom-right
PENALTY_STATE = (1, 3)  # row 1, col 3
TERMINAL_STATES = {GOAL_STATE}

def state_to_idx(row, col):
    return row * GRID_SIZE + col

def idx_to_state(idx):
    return (idx // GRID_SIZE, idx % GRID_SIZE)

# Build transition probabilities and rewards
P_gw = np.zeros((N_STATES_GW, N_ACTIONS_GW, N_STATES_GW))
R_gw = np.zeros((N_STATES_GW, N_ACTIONS_GW))

for row in range(GRID_SIZE):
    for col in range(GRID_SIZE):
        s = state_to_idx(row, col)
        
        if (row, col) in TERMINAL_STATES:
            for a in range(N_ACTIONS_GW):
                P_gw[s, a, s] = 1.0
                R_gw[s, a] = 0.0
            continue
        
        for a in range(N_ACTIONS_GW):
            dr, dc = ACTION_DELTAS[a]
            new_row = max(0, min(GRID_SIZE - 1, row + dr))
            new_col = max(0, min(GRID_SIZE - 1, col + dc))
            s_next = state_to_idx(new_row, new_col)
            P_gw[s, a, s_next] = 1.0  # deterministic
            
            if (new_row, new_col) == GOAL_STATE:
                R_gw[s, a] = 1.0
            elif (new_row, new_col) == PENALTY_STATE:
                R_gw[s, a] = -1.0
            else:
                R_gw[s, a] = -0.04

# Display the grid
print("\n📋 Grid Layout:")
print("+" + "-------+" * GRID_SIZE)
for row in range(GRID_SIZE):
    print("|", end='')
    for col in range(GRID_SIZE):
        if (row, col) == GOAL_STATE:
            print(" 🏆 G ", end='|')
        elif (row, col) == PENALTY_STATE:
            print(" ⚠️ P ", end='|')
        else:
            s = state_to_idx(row, col)
            print(f"  {s:2d}  ", end='|')
    print()
    print("+" + "-------+" * GRID_SIZE)

print("\n🏆 G = Goal (+1 reward)")
print("⚠️ P = Penalty (-1 reward)")
print("All other transitions: -0.04 reward")
print(f"γ = {GAMMA_GW}")

In [ ]:
# === Policy Evaluation on uniform random policy ===

print("="*60)
print("🎲 POLICY EVALUATION — Uniform Random Policy")
print("="*60)

# Uniform random policy: equal probability for all 4 actions
pi_uniform = np.ones((N_STATES_GW, N_ACTIONS_GW)) / N_ACTIONS_GW

V_gw_pi = np.zeros(N_STATES_GW)
n_iter_gw = 500
tol_gw = 1e-8

for iteration in range(n_iter_gw):
    V_new = np.zeros(N_STATES_GW)
    for s in range(N_STATES_GW):
        r, c = idx_to_state(s)
        if (r, c) in TERMINAL_STATES:
            V_new[s] = 0.0
            continue
        for a in range(N_ACTIONS_GW):
            V_new[s] += pi_uniform[s, a] * (R_gw[s, a] + GAMMA_GW * np.dot(P_gw[s, a], V_gw_pi))
    
    delta = np.max(np.abs(V_new - V_gw_pi))
    V_gw_pi = V_new.copy()
    if delta < tol_gw:
        print(f"✅ Converged in {iteration+1} iterations")
        break

# Display V^π as a grid
V_grid_pi = V_gw_pi.reshape(GRID_SIZE, GRID_SIZE)

print("\n📊 V^π values (uniform random policy):")
for row in range(GRID_SIZE):
    for col in range(GRID_SIZE):
        print(f"{V_grid_pi[row, col]:8.3f}", end=' ')
    print()

# Heatmap
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(V_grid_pi, cmap='RdYlGn', interpolation='nearest')

for i in range(GRID_SIZE):
    for j in range(GRID_SIZE):
        label = f"{V_grid_pi[i, j]:.3f}"
        if (i, j) == GOAL_STATE:
            label += "\n🏆"
        elif (i, j) == PENALTY_STATE:
            label += "\n⚠️"
        ax.text(j, i, label, ha='center', va='center', fontsize=12, fontweight='bold')

ax.set_xticks(range(GRID_SIZE))
ax.set_yticks(range(GRID_SIZE))
ax.set_xlabel('Column', fontsize=13)
ax.set_ylabel('Row', fontsize=13)
ax.set_title('🎲 V^π Under Uniform Random Policy', fontsize=15, fontweight='bold')
plt.colorbar(im, ax=ax, label='State Value')

plt.tight_layout()
plt.show()

print("\n💡 Under a random policy, states closer to the goal have higher values.")
print("   The penalty state pulls down values of nearby cells.")

In [ ]:
# === Value Iteration for V* with optimal policy arrows ===

print("="*60)
print("⭐ VALUE ITERATION — Finding V* and Optimal Policy")
print("="*60)

V_gw_star = np.zeros(N_STATES_GW)

for iteration in range(n_iter_gw):
    V_new = np.zeros(N_STATES_GW)
    for s in range(N_STATES_GW):
        r, c = idx_to_state(s)
        if (r, c) in TERMINAL_STATES:
            V_new[s] = 0.0
            continue
        q_vals = np.zeros(N_ACTIONS_GW)
        for a in range(N_ACTIONS_GW):
            q_vals[a] = R_gw[s, a] + GAMMA_GW * np.dot(P_gw[s, a], V_gw_star)
        V_new[s] = np.max(q_vals)
    
    delta = np.max(np.abs(V_new - V_gw_star))
    V_gw_star = V_new.copy()
    if delta < tol_gw:
        print(f"✅ Value iteration converged in {iteration+1} iterations")
        break

# Extract optimal policy
optimal_actions_gw = np.zeros(N_STATES_GW, dtype=int)
for s in range(N_STATES_GW):
    r, c = idx_to_state(s)
    if (r, c) in TERMINAL_STATES:
        continue
    q_vals = np.zeros(N_ACTIONS_GW)
    for a in range(N_ACTIONS_GW):
        q_vals[a] = R_gw[s, a] + GAMMA_GW * np.dot(P_gw[s, a], V_gw_star)
    optimal_actions_gw[s] = np.argmax(q_vals)

V_grid_star = V_gw_star.reshape(GRID_SIZE, GRID_SIZE)

print("\n📊 V* values and optimal actions:")
for row in range(GRID_SIZE):
    for col in range(GRID_SIZE):
        s = state_to_idx(row, col)
        if (row, col) in TERMINAL_STATES:
            print(f"  {'GOAL':>8}", end=' ')
        else:
            print(f"  {V_grid_star[row,col]:6.3f}{ACTION_NAMES_GW[optimal_actions_gw[s]]}", end=' ')
    print()

# Visualization with arrows
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: V* heatmap
im1 = axes[0].imshow(V_grid_star, cmap='RdYlGn', interpolation='nearest')
for i in range(GRID_SIZE):
    for j in range(GRID_SIZE):
        label = f"{V_grid_star[i, j]:.3f}"
        if (i, j) == GOAL_STATE:
            label += "\n🏆"
        elif (i, j) == PENALTY_STATE:
            label += "\n⚠️"
        axes[0].text(j, i, label, ha='center', va='center', fontsize=11, fontweight='bold')

axes[0].set_xticks(range(GRID_SIZE))
axes[0].set_yticks(range(GRID_SIZE))
axes[0].set_xlabel('Column', fontsize=13)
axes[0].set_ylabel('Row', fontsize=13)
axes[0].set_title('⭐ Optimal State Values V*', fontsize=14, fontweight='bold')
plt.colorbar(im1, ax=axes[0], label='V*')

# Right: Policy arrows on heatmap
im2 = axes[1].imshow(V_grid_star, cmap='RdYlGn', interpolation='nearest', alpha=0.6)

# Arrow deltas for visualization (dx, dy) — note: y is inverted for image coords
arrow_dx = [0, 0, -0.3, 0.3]   # Up, Down, Left, Right -> column delta
arrow_dy = [-0.3, 0.3, 0, 0]   # Up, Down, Left, Right -> row delta

for i in range(GRID_SIZE):
    for j in range(GRID_SIZE):
        s = state_to_idx(i, j)
        if (i, j) in TERMINAL_STATES:
            axes[1].text(j, i, '🏆', ha='center', va='center', fontsize=20)
            continue
        elif (i, j) == PENALTY_STATE:
            axes[1].text(j, i + 0.3, '⚠️', ha='center', va='center', fontsize=12)
        
        a = optimal_actions_gw[s]
        axes[1].annotate('', xy=(j + arrow_dx[a], i + arrow_dy[a]),
                        xytext=(j, i),
                        arrowprops=dict(arrowstyle='->', color='darkblue',
                                       lw=2.5, mutation_scale=20))
        axes[1].text(j, i, ACTION_NAMES_GW[a], ha='center', va='center',
                    fontsize=16, fontweight='bold', color='darkblue')

axes[1].set_xticks(range(GRID_SIZE))
axes[1].set_yticks(range(GRID_SIZE))
axes[1].set_xlabel('Column', fontsize=13)
axes[1].set_ylabel('Row', fontsize=13)
axes[1].set_title('🎯 Optimal Policy π*', fontsize=14, fontweight='bold')
plt.colorbar(im2, ax=axes[1], label='V*')

plt.suptitle('🗺️ Grid World: Optimal Solution via Bellman Equations',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 The optimal policy guides the agent toward the goal while avoiding the penalty.")
print("   Arrows show the best direction to move from each cell.")
print("   Cells near the goal have highest V* (green), cells near penalty are lower (red).")

---

## 🖼️ Section 8: Bellman Equations for Image Processing

### Connecting RL to Image Processing

How do the Bellman equations apply to **image processing**? Consider this mapping:

| RL Concept | Image Processing Interpretation |
|-----------|--------------------------------|
| **State** $s$ | Current image quality level |
| **Action** $a$ | Image operation (enhance, denoise, sharpen, ...) |
| **Reward** $R(s,a)$ | Quality improvement from applying operation |
| **Transition** $P(s'\mid s,a)$ | Probability of reaching a quality level after operation |
| **$V(s)$** | Expected total quality improvement possible from current state |
| **$Q(s,a)$** | Expected improvement from taking a specific action in current state |
| **Optimal policy** $\pi^*$ | Best sequence of operations for any image quality |

### Example: Image Quality Enhancement MDP

- **States**: {Poor, Medium, Good} — representing image quality levels
- **Actions**: {Enhance, No-op}
- The "Enhance" action can improve quality but sometimes fails
- The "No-op" action keeps quality the same (or slightly degrades)

In [ ]:
# === Image Quality Enhancement MDP ===

print("="*60)
print("🖼️ IMAGE QUALITY ENHANCEMENT MDP")
print("="*60)

iq_states = ['Poor', 'Medium', 'Good']
iq_actions = ['Enhance', 'No-op']
n_iq_s = len(iq_states)
n_iq_a = len(iq_actions)
gamma_iq = 0.9

# Transition probabilities
P_iq = np.zeros((n_iq_s, n_iq_a, n_iq_s))

# Enhance action
P_iq[0, 0] = [0.3, 0.6, 0.1]   # Poor -> Enhance: 60% chance of Medium, 10% Good
P_iq[1, 0] = [0.05, 0.35, 0.6]  # Medium -> Enhance: 60% chance of Good
P_iq[2, 0] = [0.0, 0.1, 0.9]   # Good -> Enhance: mostly stays Good

# No-op action
P_iq[0, 1] = [0.9, 0.1, 0.0]   # Poor -> No-op: mostly stays Poor
P_iq[1, 1] = [0.2, 0.7, 0.1]   # Medium -> No-op: might degrade
P_iq[2, 1] = [0.05, 0.25, 0.7]  # Good -> No-op: might degrade

# Rewards
R_iq = np.array([
    [2.0, 0.0],    # Poor: Enhance gives reward, No-op gives nothing
    [3.0, 0.5],    # Medium: Enhance gives more reward
    [1.0, 1.0]     # Good: Both actions give moderate reward (maintaining quality)
])

print(f"\n📋 States: {iq_states}")
print(f"📋 Actions: {iq_actions}")
print(f"📋 γ = {gamma_iq}")

print("\n📊 Transition Probabilities:")
for s in range(n_iq_s):
    for a in range(n_iq_a):
        probs = ', '.join([f'{iq_states[sp]}:{P_iq[s,a,sp]:.2f}' for sp in range(n_iq_s)])
        print(f"  P(·|{iq_states[s]}, {iq_actions[a]}) = [{probs}]")

print("\n📊 Rewards R(s,a):")
for s in range(n_iq_s):
    for a in range(n_iq_a):
        print(f"  R({iq_states[s]}, {iq_actions[a]}) = {R_iq[s,a]:.1f}")

# Value Iteration to find optimal policy
V_iq = np.zeros(n_iq_s)
for i in range(500):
    V_new = np.zeros(n_iq_s)
    for s in range(n_iq_s):
        q_vals = np.array([R_iq[s, a] + gamma_iq * np.dot(P_iq[s, a], V_iq) for a in range(n_iq_a)])
        V_new[s] = np.max(q_vals)
    if np.max(np.abs(V_new - V_iq)) < 1e-8:
        print(f"\n✅ Value iteration converged in {i+1} iterations")
        break
    V_iq = V_new

# Compute Q* and extract policy
Q_iq = np.zeros((n_iq_s, n_iq_a))
for s in range(n_iq_s):
    for a in range(n_iq_a):
        Q_iq[s, a] = R_iq[s, a] + gamma_iq * np.dot(P_iq[s, a], V_iq)

print("\n" + "="*50)
print("📊 RESULTS: Image Quality Enhancement MDP")
print("="*50)

print("\n📊 Optimal Values V*(s):")
for s in range(n_iq_s):
    print(f"  V*({iq_states[s]:>6}) = {V_iq[s]:.4f}")

print("\n📊 Q*(s,a) Values:")
print(f"  {'State':<10}", end='')
for a in range(n_iq_a):
    print(f"{iq_actions[a]:<15}", end='')
print(f"{'Best Action':<15}")
print("  " + "-"*45)
for s in range(n_iq_s):
    print(f"  {iq_states[s]:<10}", end='')
    for a in range(n_iq_a):
        print(f"{Q_iq[s,a]:<15.4f}", end='')
    best_a = np.argmax(Q_iq[s])
    print(f"{iq_actions[best_a]:<15} ⬅️")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# V* bar chart
colors_iq = ['#e74c3c', '#f39c12', '#27ae60']
axes[0].bar(iq_states, V_iq, color=colors_iq, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Optimal Value V*', fontsize=12)
axes[0].set_title('📊 V* per Quality State', fontsize=14, fontweight='bold')
for i, v in enumerate(V_iq):
    axes[0].text(i, v + 0.2, f'{v:.2f}', ha='center', fontsize=12, fontweight='bold')

# Q* heatmap
im = axes[1].imshow(Q_iq, cmap='YlOrRd', aspect='auto')
axes[1].set_xticks(range(n_iq_a))
axes[1].set_xticklabels(iq_actions, fontsize=12)
axes[1].set_yticks(range(n_iq_s))
axes[1].set_yticklabels(iq_states, fontsize=12)
axes[1].set_title('🎯 Q*(state, action)', fontsize=14, fontweight='bold')
for i in range(n_iq_s):
    for j in range(n_iq_a):
        axes[1].text(j, i, f'{Q_iq[i,j]:.2f}', ha='center', va='center',
                     fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[1], label='Q-Value')

# Optimal policy visualization
opt_policy_iq = np.zeros((n_iq_s, n_iq_a))
for s in range(n_iq_s):
    opt_policy_iq[s, np.argmax(Q_iq[s])] = 1.0

im2 = axes[2].imshow(opt_policy_iq, cmap='Blues', aspect='auto', vmin=0, vmax=1)
axes[2].set_xticks(range(n_iq_a))
axes[2].set_xticklabels(iq_actions, fontsize=12)
axes[2].set_yticks(range(n_iq_s))
axes[2].set_yticklabels(iq_states, fontsize=12)
axes[2].set_title('🎯 Optimal Policy π*', fontsize=14, fontweight='bold')
for i in range(n_iq_s):
    for j in range(n_iq_a):
        txt = '✓' if opt_policy_iq[i,j] > 0.5 else ''
        axes[2].text(j, i, txt, ha='center', va='center', fontsize=20,
                     fontweight='bold', color='white' if opt_policy_iq[i,j] > 0.5 else 'lightgray')
plt.colorbar(im2, ax=axes[2], label='Probability')

plt.suptitle('🖼️ Image Quality Enhancement — Bellman Equation Solution',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n🔑 Interpretation:")
print("─" * 60)
for s in range(n_iq_s):
    best_a = np.argmax(Q_iq[s])
    print(f"  • When image is {iq_states[s]:>6}: optimal action is '{iq_actions[best_a]}'")
    print(f"    Expected total future improvement: {V_iq[s]:.2f}")

print("\n💡 The Bellman equations tell us that:")
print("   - 'Enhance' is optimal for Poor and Medium quality images")
print("   - This makes intuitive sense: invest in improvement when quality is low")
print("   - The value function quantifies exactly HOW MUCH improvement to expect")

---

## 📝 Summary

### The Four Bellman Equations at a Glance

| Equation | Formula | Type | Solution Method |
|----------|---------|------|-----------------|
| **Bellman Expectation for $V^\pi$** | $V^\pi(s) = \sum_a \pi(a{\mid}s)\left[R(s,a) + \gamma \sum_{s'} P(s'{\mid}s,a) V^\pi(s')\right]$ | Linear | Matrix inversion or iterative evaluation |
| **Bellman Expectation for $Q^\pi$** | $Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'{\mid}s,a) \sum_{a'} \pi(a'{\mid}s') Q^\pi(s',a')$ | Linear | Matrix inversion or iterative evaluation |
| **Bellman Optimality for $V^*$** | $V^*(s) = \max_a \left[R(s,a) + \gamma \sum_{s'} P(s'{\mid}s,a) V^*(s')\right]$ | Nonlinear | Value iteration |
| **Bellman Optimality for $Q^*$** | $Q^*(s,a) = R(s,a) + \gamma \sum_{s'} P(s'{\mid}s,a) \max_{a'} Q^*(s',a')$ | Nonlinear | Q-value iteration |

### 🔑 Key Insights

1. **Recursion is fundamental**: The Bellman equations express the value of a state in terms of values of successor states — this is the mathematical engine of RL.

2. **Expectation vs. Optimality**: The expectation equations evaluate a *given* policy (using $\sum_a \pi(a|s)$), while the optimality equations find the *best* policy (using $\max_a$).

3. **V and Q are linked**: $V^\pi(s) = \sum_a \pi(a|s) Q^\pi(s,a)$ and $Q^\pi(s,a) = R(s,a) + \gamma \sum_{s'} P(s'|s,a) V^\pi(s')$.

4. **Q\* gives the policy directly**: Once you know $Q^*(s,a)$, the optimal policy is simply $\pi^*(s) = \arg\max_a Q^*(s,a)$ — no model needed!

5. **For image processing**: The Bellman equations help determine the optimal sequence of image operations by recursively computing the expected quality improvement.

---

### ➡️ Next: Module 3.4 — Value Functions

In the next module, we'll dive deeper into value function estimation methods, including Monte Carlo and Temporal Difference learning, which approximate the Bellman equations when the MDP dynamics are unknown.

---

*© 2024 FlashVision — RL for Image Processing Course*